# 03 — Silver Layer Prototype

**Goal:** Prototype the Bronze → Silver transformation interactively before relying on the production script `spark/silver/validator.py`.

This notebook applies four of the ten critical/warning rules from `docs/silver-layer-specification.md` to the raw January 2024 trip data, splits records into a clean Silver dataset and per-rule quarantine datasets, and generates a JSON quality report.

> **Note:** this notebook is a scoped-down prototype (4 rules, January only). The full rule set — including SLV-007 (negative total), SLV-008/009 (reporting-period flags), SLV-010 (distance-outlier threshold), and missing-value monitoring — plus all 6 available months, is implemented in `spark/silver/validator.py`, which is now the canonical Silver pipeline. Treat the numbering below as illustrative; `docs/silver-layer-specification.md` is the source of truth.

| Rule | Condition | Failure → |
|---|---|---|
| SLV-003 | `tpep_dropoff_datetime > tpep_pickup_datetime` | quarantine (`invalid_timestamps`) |
| SLV-004 | `trip_distance >= 0` | quarantine (`distance_outliers`) |
| SLV-005 | `passenger_count > 0` | quarantine (`invalid_passenger_count`) |
| SLV-006 | `fare_amount >= 0` | quarantine (`negative_fares`) |

**Input:** `data/raw/yellow_taxi/2024-01.parquet`
**Outputs:** `data/silver/trips/`, `data/silver/quarantine/`, `docs/quality-reports/`

## Step 1 — Load the Bronze dataset

In [1]:
import json
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd().parent
RAW_PATH = BASE_DIR / "data" / "raw" / "yellow_taxi" / "2024-01.parquet"
SILVER_DIR = BASE_DIR / "data" / "silver" / "trips"
QUARANTINE_DIR = BASE_DIR / "data" / "silver" / "quarantine"
REPORTS_DIR = BASE_DIR / "data" / "reports"

df = pd.read_parquet(RAW_PATH, engine="pyarrow")
print("Loaded:", df.shape)

Loaded: (2964624, 19)


## Step 2 — Build validation masks

One boolean mask per rule, computed over the whole DataFrame. These get combined later to decide which rows make it into the Silver dataset.

In [2]:
valid_timestamp = df["tpep_dropoff_datetime"] > df["tpep_pickup_datetime"]
valid_passenger = df["passenger_count"] > 0
valid_fare = df["fare_amount"] >= 0
valid_distance = df["trip_distance"] >= 0

print("valid_timestamp :", valid_timestamp.sum(), "/", len(df))
print("valid_passenger :", valid_passenger.sum(), "/", len(df))
print("valid_fare      :", valid_fare.sum(), "/", len(df))
print("valid_distance  :", valid_distance.sum(), "/", len(df))

valid_timestamp : 2963754 / 2964624
valid_passenger : 2792997 / 2964624
valid_fare      : 2927176 / 2964624
valid_distance  : 2964624 / 2964624


## Step 3 — Build quarantine datasets

Each failing rule gets its own quarantine dataset rather than a single bucket, so downstream investigation can target one failure mode at a time (e.g. "why are 37K fares negative?").

In [3]:
negative_fares = df[df["fare_amount"] < 0]
invalid_passengers = df[df["passenger_count"] <= 0]
invalid_timestamps = df[df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]]

print("negative_fares     :", len(negative_fares))
print("invalid_passengers :", len(invalid_passengers))
print("invalid_timestamps :", len(invalid_timestamps))

negative_fares     : 37448
invalid_passengers : 31465
invalid_timestamps : 56


## Step 4 — Build the Silver dataset

Only rows passing *all four* masks survive. This is a strict AND, matching `docs/silver-layer-specification.md`: a record with a perfectly fine fare but a broken timestamp still doesn't belong in Silver.

In [4]:
silver_df = df[valid_timestamp & valid_passenger & valid_fare & valid_distance]

print("records_processed  :", len(df))
print("records_valid      :", len(silver_df))
print("records_quarantined:", len(df) - len(silver_df))

records_processed  : 2964624
records_valid      : 2756936
records_quarantined: 207688


## Step 5 — Persist outputs to disk

Writes the Silver dataset and each quarantine dataset to Parquet, creating the output directories if they don't already exist.

In [5]:
SILVER_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

silver_df.to_parquet(SILVER_DIR / "silver_trips_2024_01.parquet", engine="pyarrow")
negative_fares.to_parquet(QUARANTINE_DIR / "negative_fares.parquet", engine="pyarrow")
invalid_passengers.to_parquet(QUARANTINE_DIR / "invalid_passengers.parquet", engine="pyarrow")
invalid_timestamps.to_parquet(QUARANTINE_DIR / "invalid_timestamps.parquet", engine="pyarrow")

print("Silver + quarantine files written.")

Silver + quarantine files written.


## Step 6 — Generate the quality report

A machine-readable summary of this run, written to `data/reports/`. Automated pipelines (and later, dashboards) can read this instead of re-deriving counts from the data.

In [6]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

report = {
    "records_processed": int(len(df)),
    "records_valid": int(len(silver_df)),
    "records_quarantined": int(len(df) - len(silver_df)),
    "negative_fares": int(len(negative_fares)),
    "invalid_passengers": int(len(invalid_passengers)),
    "invalid_timestamps": int(len(invalid_timestamps)),
    "negative_distances": int((df["trip_distance"] < 0).sum()),
}

with open(REPORTS_DIR / "quality_report_2024_01.json", "w") as f:
    json.dump(report, f, indent=2)

report

{'records_processed': 2964624,
 'records_valid': 2756936,
 'records_quarantined': 207688,
 'negative_fares': 37448,
 'invalid_passengers': 31465,
 'invalid_timestamps': 56,
 'negative_distances': 0}

## Summary & Next Steps

This notebook prototyped 4 of the 10 Silver rules against a single month. The production pipeline, `spark/silver/validator.py`, now implements the full rule set (reject / quarantine / flag / monitor tiers) across all 6 available months, reading from `data/bronze/` (built by `ingestion/run_ingestion.py`) instead of `data/raw/` directly. See `docs/implementation-plan.md` for current status.

Its outputs, per month:

- `data/silver/trips/silver_trips_{year}_{month}.parquet`
- `data/silver/quarantine/{invalid_timestamps,distance_outliers,invalid_passenger_count,negative_fares,negative_total_amounts}_{year}_{month}.parquet`
- `docs/quality-reports/quality-report-{year}-{month}.json`

Prototype new rules or thresholds here first, then promote them into `validator.py` so the production pipeline stays in sync.

**Next:** Gold-layer aggregations (daily revenue, hourly demand, zone-level stats, congestion metrics) reading from `data/silver/trips/`.